In [ ]:
import gradio as gr
import numpy as np
import joblib
from PIL import Image
import math
import re

# 1. Safe loading of pre-trained models
try:
    models = {
        "Logistic Regression": joblib.load('logistic_regression_model.pkl'),
        "Naive Bayes": joblib.load('naive_bayes_model.pkl'),
        "Support Vector Machine (SVM)": joblib.load('svm_model.pkl')
    }
except FileNotFoundError:
    print("Error: Model files not found! Make sure to export them via joblib first.")

def extract_grayscale_image(sketchpad_data):
    """
    Safely converts Gradio canvas data into a grayscale PIL Image, 
    smoothly handling dictionary vs raw array format variations.
    """
    if sketchpad_data is None:
        return None
        
    # Handle dictionary format (newer Gradio versions)
    if isinstance(sketchpad_data, dict):
        if "composite" in sketchpad_data and sketchpad_data["composite"] is not None:
            img_data = sketchpad_data["composite"]
        elif "layers" in sketchpad_data and len(sketchpad_data["layers"]) > 0:
            img_data = sketchpad_data["layers"][0]
        else:
            img_data = next((v for v in sketchpad_data.values() if v is not None), None)
    else:
        # Handle raw numpy array format (older Gradio versions)
        img_data = sketchpad_data
        
    if img_data is None:
        return None
        
    # Convert extracted structure to a grayscale PIL Image
    if isinstance(img_data, np.ndarray):
        return Image.fromarray(img_data.astype('uint8')).convert('L')
    elif isinstance(img_data, Image.Image):
        return img_data.convert('L')
    return None

def predict_digit(model_choice, sketchpad_data):
    # Extract canvas drawing
    img_gray = extract_grayscale_image(sketchpad_data)
    if img_gray is None:
        return "Please draw a digit first."
        
    selected_model = models[model_choice]
    
    # Start by assuming a baseline MNIST dimension of 28x28 (784 features)
    target_features = 784
    side_length = 28
    
    # Self-healing loop: Attempt execution, capture expectations, reshape if required
    for attempt in range(2):
        img_resized = img_gray.resize((side_length, side_length))
        img_array = np.array(img_resized)
        
        # Correct canvas contrast polarity if drawing background is light 
        # (MNIST models expect dark backgrounds with bright strokes)
        corners = [img_array[0, 0], img_array[0, -1], img_array[-1, 0], img_array[-1, -1]]
        if np.mean(corners) > 127:
            img_array = 255 - img_array
            
        # Flatten matrix into a 1D vector matching target features
        pixel_features = img_array.reshape(1, target_features)
        
        try:
            # Attempt inference
            prediction = selected_model.predict(pixel_features)[0]
            return f"Model: {model_choice}\nResolution: {side_length}x{side_length} ({target_features} features)\n\nPredicted Digit: {prediction}"
            
        except ValueError as e:
            error_msg = str(e)
            # Use regex to check if scikit-learn explicitly demands a different feature size
            match = re.search(r"expecting\s+(\d+)\s+features", error_msg)
            if match and attempt == 0:
                # Read the specific feature count, calculate square dimension size, and retry
                target_features = int(match.group(1))
                side_length = int(math.sqrt(target_features))
                continue 
            else:
                return f"Prediction Error: {error_msg}"
                
    return "Failed to match the model's expected dimension shape configuration."

# 2. Build layout interface safely (removing strict crop parameters to avoid version conflicts)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔢 Universal Digit Recognition Frontend")
    gr.Markdown(
        "Draw a single digit (0-9) in the center of the grid, select your preferred "
        "trained model from the dropdown menu, and hit recognize!"
    )
    
    with gr.Row():
        with gr.Column():
            canvas = gr.Sketchpad(label="Draw here", type="numpy")
            model_dropdown = gr.Dropdown(
                choices=["Logistic Regression", "Naive Bayes", "Support Vector Machine (SVM)"],
                value="Logistic Regression",
                label="Select Classification Model"
            )
            submit_btn = gr.Button("Recognize Digit", variant="primary")
            clear_btn = gr.Button("Clear Canvas")
            
        with gr.Column():
            output_text = gr.Label(label="Prediction Output Summary")

    # Connect components
    submit_btn.click(
        fn=predict_digit, 
        inputs=[model_dropdown, canvas], 
        outputs=output_text
    )
    clear_btn.click(lambda: None, None, canvas)

if __name__ == "__main__":
    demo.launch(inline=False, inbrowser=True)